# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:

import pandas as pd
import numpy as np
from io import StringIO
from IPython.display import display

# Bestandspad naar de oorspronkelijke CSV-dataset
bestandspad = "SolarNL_exploration_data.csv"

# CSV-bestand inlezen
with open(bestandspad, "r", encoding="utf-8-sig") as bestand:
    regels = bestand.readlines()

# Afzonderlijke InfluxDB-tabellen herkennen
tabellen = []
huidige_tabel = []

for regel in regels:

    if regel.startswith("#datatype"):
        if huidige_tabel:
            tabellen.append(huidige_tabel)
        huidige_tabel = []

    elif regel.startswith("#") or not regel.strip():
        continue

    else:
        huidige_tabel.append(regel)

if huidige_tabel:
    tabellen.append(huidige_tabel)

# Tabellen omzetten naar Pandas DataFrames
dataframes = []

for tabel in tabellen:

    df = pd.read_csv(
        StringIO("".join(tabel)),
        sep=";"
    )

    # Lege kolommen en overbodige metadata verwijderen
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    df = df.drop(
        columns=["result", "table", "_start", "_stop"],
        errors="ignore"
    )

    # Volledig lege rijen verwijderen
    df = df.dropna(how="all")

    # Index opnieuw instellen
    df = df.reset_index(drop=True)

    dataframes.append(df)

# ============================================================
# WEERDATA SAMENVOEGEN
# ============================================================

df_weather = pd.concat(
    [
        df for df in dataframes
        if "Humidity-PVMET500" in df.columns
    ],
    ignore_index=True
)

# ============================================================
# ZONNEPANEELDATA SAMENVOEGEN
# ============================================================

# Ook tabellen met uitsluitend Pt100-metingen meenemen
solar_meetkolommen = [
    "ff", "impp", "isc", "vmpp", "voc"
]

solar_tabellen = [
    df for df in dataframes
    if (
        any(kolom in df.columns for kolom in solar_meetkolommen)
        or any(kolom.startswith("Pt100") for kolom in df.columns)
    )
    and (
        "pvdeviceId" in df.columns
        or "label" in df.columns
    )
]

df_solar = pd.concat(
    solar_tabellen,
    ignore_index=True
)

# ============================================================
# TIJDSTEMPELS EN MEETWAARDEN CONVERTEREN
# ============================================================

for df in [df_weather, df_solar]:

    df["_time"] = pd.to_datetime(
        df["_time"],
        utc=True,
        errors="coerce"
    )

    # Kolommen die geen numerieke meetwaarden bevatten
    metadata_kolommen = [
        "_time", "_measurement", "label", "pvdeviceId"
    ]

    for kolom in df.columns:
        if kolom not in metadata_kolommen:
            df[kolom] = pd.to_numeric(
                df[kolom],
                errors="coerce"
            )

    # Metingen chronologisch sorteren
    df.sort_values("_time", inplace=True)
    df.reset_index(drop=True, inplace=True)

# ============================================================
# OORSPRONKELIJKE ZONNEPANEELDATA BEWAREN
# ============================================================

# Deze kopie blijft beschikbaar voor latere bewerkingen
df_solar_raw = df_solar.copy()

# ============================================================
# CONTROLE
# ============================================================

print("Datasets succesvol ingeladen!")
print(f"Weerdataset:         {df_weather.shape}")
print(f"Zonnepaneeldataset:  {df_solar.shape}")

print(f"\nAantal InfluxDB-tabellen: {len(dataframes)}")
print(f"Zonnepaneeltabellen:       {len(solar_tabellen)}")


# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ============================================================
# 1. ZONNEPANEELDATA OPSCHONEN
# ============================================================

# Begin opnieuw met de oorspronkelijke data,
# zodat eerder verwijderde meetwaarden terugkomen
df_solar = df_solar_raw.copy()

# Oorspronkelijk aantal zonnepaneelmetingen
aantal_voor = len(df_solar)

# Alleen de oorspronkelijke Pt100-sensorkolommen herkennen
# De samengevoegde kolom 'Pt100' wordt uitgesloten
pt100_kolommen = [
    kolom for kolom in df_solar.columns
    if kolom.startswith("Pt100") and kolom != "Pt100"
]

# Pt100-sensorkolommen samenvoegen indien aanwezig
if pt100_kolommen:

    pt100_samengevoegd = (
        df_solar[pt100_kolommen]
        .bfill(axis=1)
        .iloc[:, 0]
    )

    # Eventueel bestaande Pt100-waarden behouden
    if "Pt100" in df_solar.columns:
        df_solar["Pt100"] = (
            df_solar["Pt100"]
            .combine_first(pt100_samengevoegd)
        )
    else:
        df_solar["Pt100"] = pt100_samengevoegd

    # Oorspronkelijke sensorkolommen verwijderen
    df_solar = df_solar.drop(columns=pt100_kolommen)

# Meetvariabelen
meetkolommen = [
    "Pt100",
    "ff",
    "impp",
    "isc",
    "vmpp",
    "voc"
]

# Meetwaarden omzetten naar numerieke waarden
for kolom in meetkolommen:
    df_solar[kolom] = pd.to_numeric(
        df_solar[kolom],
        errors="coerce"
    )

# Alleen meetrijen verwijderen waarbij
# ALLE zes meetvariabelen ontbreken
df_solar = df_solar.dropna(
    subset=meetkolommen,
    how="all"
).copy()

# Rijen zonder tijdstip of paneelidentificatie verwijderen
df_solar = df_solar.dropna(
    subset=["_time", "label"]
).copy()

# Chronologisch sorteren en index herstellen
df_solar = df_solar.sort_values(
    ["_time", "pvdeviceId"]
).reset_index(drop=True)

# Resultaten controleren
aantal_na = len(df_solar)

print("Opschonen zonnepaneeldata voltooid!")
print(f"Metingen vóór opschonen: {aantal_voor}")
print(f"Metingen na opschonen:   {aantal_na}")
print(f"Verwijderde metingen:    {aantal_voor - aantal_na}")


# ============================================================
# 2. AANTAL MEETRIJEN PER ZONNEPANEEL
# ============================================================

# Aantal behouden meetrijen per zonnepaneel berekenen
metingen_per_paneel = (
    df_solar.groupby("label")
    .size()
    .to_frame("Aantal metingen")
)

# Figuur met tabel links en grafiek rechts
fig, (ax_tabel, ax_grafiek) = plt.subplots(
    1, 2,
    figsize=(9, 3.5),
    gridspec_kw={"width_ratios": [1.2, 2.5]}
)

# Transparante achtergrond
fig.patch.set_facecolor("none")
ax_tabel.set_facecolor("none")
ax_grafiek.set_facecolor("none")


# -----------------------------
# Links: tabel
# -----------------------------

ax_tabel.axis("off")

tabel = ax_tabel.table(
    cellText=[
        [paneel, aantal]
        for paneel, aantal in zip(
            metingen_per_paneel.index,
            metingen_per_paneel["Aantal metingen"]
        )
    ],
    colLabels=["Zonnepaneel", "Metingen"],
    colWidths=[0.65, 0.35],
    cellLoc="center",
    loc="center"
)

tabel.auto_set_font_size(False)
tabel.set_fontsize(9)
tabel.scale(1.1, 1.5)

# Tabelcellen transparant maken
for (rij, kolom), cel in tabel.get_celld().items():

    cel.set_facecolor("none")
    cel.set_edgecolor("gray")
    cel.set_linewidth(0.5)

    if rij == 0:
        cel.set_text_props(weight="bold")


# -----------------------------
# Rechts: staafdiagram
# -----------------------------

ax_grafiek.barh(
    metingen_per_paneel.index,
    metingen_per_paneel["Aantal metingen"],
    color="steelblue",
    height=0.65
)

ax_grafiek.invert_yaxis()

ax_grafiek.set_title(
    "Aantal metingen per zonnepaneel",
    fontsize=11
)

ax_grafiek.set_xlabel(
    "Aantal metingen",
    fontsize=9
)

ax_grafiek.tick_params(
    axis="both",
    labelsize=9
)

ax_grafiek.grid(
    axis="x",
    linestyle="--",
    alpha=0.2
)

ax_grafiek.set_axisbelow(True)

# Meetwaarden naast de staven
for i, aantal in enumerate(
    metingen_per_paneel["Aantal metingen"]
):
    ax_grafiek.text(
        aantal + 3,
        i,
        str(aantal),
        va="center",
        fontsize=9
    )

ax_grafiek.set_xlim(
    0,
    metingen_per_paneel["Aantal metingen"].max() * 1.15
)

# Onnodige randen verwijderen
ax_grafiek.spines["top"].set_visible(False)
ax_grafiek.spines["right"].set_visible(False)

# Indeling
fig.subplots_adjust(
    left=0.04,
    right=0.97,
    top=0.88,
    bottom=0.15,
    wspace=0.55
)

plt.show()


# ============================================================
# 3. BESCHIKBARE METINGEN PER VARIABELE
# ============================================================

print("\nBeschikbare metingen per variabele en zonnepaneel:")

beschikbare_metingen = (
    df_solar.groupby("label")[meetkolommen]
    .count()
)

display(beschikbare_metingen)


# ============================================================
# 4. EERSTE TIEN RIJEN CONTROLEREN
# ============================================================

print("\nEerste 10 rijen opgeschoonde zonnepaneeldata:")
display(df_solar.head(10))


# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:

# Meetgegevens chronologisch sorteren
df_overzicht = df_solar.sort_values(
    ["label", "_time"]
).copy()

# Tijdsverschil tussen opeenvolgende metingen berekenen
df_overzicht["Tijdsverschil"] = (
    df_overzicht.groupby("label")["_time"].diff()
)

# Overzicht per zonnepaneel
tabel_tijd = df_overzicht.groupby("label").agg(
    Aantal_metingen=("_time", "size"),
    Eerste_meting=("_time", "min"),
    Laatste_meting=("_time", "max"),
    Mediaan_interval=("Tijdsverschil", "median")
)

# Mediaan interval omzetten naar minuten
tabel_tijd["Mediaan_interval"] = (
    tabel_tijd["Mediaan_interval"]
    .dt.total_seconds()
    .div(60)
    .round(1)
)

# Kolomnamen aanpassen
tabel_tijd.columns = [
    "Aantal metingen",
    "Eerste meting",
    "Laatste meting",
    "Interval (min)"
]

# Datums leesbaar maken
for kolom in ["Eerste meting", "Laatste meting"]:
    tabel_tijd[kolom] = (
        tabel_tijd[kolom]
        .dt.strftime("%d-%m-%Y %H:%M")
    )

# Tabel tonen
display(tabel_tijd)


# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Meetvariabelen en grafiektitels
variabelen = {
    "Pt100": "Paneeltemperatuur (°C)",
    "ff": "Fill factor",
    "impp": "Stroom bij maximaal vermogen (A)",
    "isc": "Kortsluitstroom (A)",
    "vmpp": "Spanning bij maximaal vermogen (V)",
    "voc": "Openklemspanning (V)"
}

# Zonnepanelen op nummer sorteren
panelen = sorted(
    df_solar["label"].dropna().unique(),
    key=lambda naam: int(naam.split("-")[-1])
)

# Matrix met 3 rijen en 2 kolommen
fig, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(14, 12),
    constrained_layout=True
)

# Kleuren
kleur = "#4682B4"

# Iedere meetvariabele plotten
for ax, (kolom, titel) in zip(
    axes.flat,
    variabelen.items()
):

    sns.boxplot(
        data=df_solar,
        x="label",
        y=kolom,
        order=panelen,
        ax=ax,
        color=kleur,
        width=0.6,
        linewidth=1,
        fliersize=3
    )

    # Grafiekopmaak
    ax.set_title(
        titel,
        fontsize=12,
        fontweight="bold",
        pad=12
    )

    ax.set_xlabel("")
    ax.set_ylabel(titel, fontsize=10)

    # Kortere labels: PvDevice-1 -> PV1
    ax.set_xticks(range(len(panelen)))
    ax.set_xticklabels(
        [f"PV{i+1}" for i in range(len(panelen))],
        fontsize=9
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.25
    )

    ax.set_axisbelow(True)

    sns.despine(ax=ax)

# Algemene titel
fig.suptitle(
    "Verdeling van de meetwaarden per zonnepaneel",
    fontsize=16,
    fontweight="bold"
)

plt.show()


# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:

import pandas as pd
import plotly.graph_objects as go

# Meetvariabelen en bijbehorende titels
variabelen = {
    "Pt100": ("Paneeltemperatuur", "Temperatuur (°C)"),
    "ff": ("Fill factor", "Fill factor"),
    "impp": ("Stroom bij maximaal vermogen", "Stroom (A)"),
    "isc": ("Kortsluitstroom", "Stroom (A)"),
    "vmpp": ("Spanning bij maximaal vermogen", "Spanning (V)"),
    "voc": ("Openklemspanning", "Spanning (V)"),
    "CMP-10": ("CMP-10 metingen", "CMP-10 (eenheid onbekend)")
}

# Vaste kleuren per zonnepaneel
paneelkleuren = {
    "PvDevice-1": "#1565C0",
    "PvDevice-2": "#42A5F5",
    "PvDevice-3": "#2E7D32",
    "PvDevice-4": "#66BB6A",
    "PvDevice-5": "#00897B",
    "PvDevice-6": "#80CBC4",
    "PvDevice-7": "#E65100",
    "PvDevice-8": "#8E24AA"
}

# Data voorbereiden
df_tijd = df_solar.copy()

df_tijd["_time"] = pd.to_datetime(
    df_tijd["_time"], utc=True
)

# Panelen sorteren
panelen = sorted(
    df_tijd["label"].dropna().unique(),
    key=lambda naam: int(naam.split("-")[-1])
)

# Voor iedere variabele een onafhankelijke figuur
for variabele, (titel, eenheid) in variabelen.items():

    fig = go.Figure()

    # CMP-10 als één gezamenlijke tijdreeks weergeven
    if variabele == "CMP-10":

        data = df_tijd[["_time", "CMP-10"]].copy()

        data["CMP-10"] = pd.to_numeric(
            data["CMP-10"], errors="coerce"
        )

        data = data.dropna(
            subset=["_time", "CMP-10"]
        )

        # Eén waarde per tijdstip, om overlappende
        # registraties van meerdere panelen te voorkomen
        data = (
            data.groupby("_time", as_index=False)["CMP-10"]
            .median()
            .sort_values("_time")
        )

        # Meetgaten bepalen
        tijdverschil = (
            data["_time"]
            .diff()
            .dt.total_seconds()
            .div(60)
        )

        x_waarden = []
        y_waarden = []

        for tijd, waarde, verschil in zip(
            data["_time"],
            data["CMP-10"],
            tijdverschil
        ):

            if pd.notna(verschil) and verschil > 20:
                x_waarden.append(tijd)
                y_waarden.append(None)

            x_waarden.append(tijd)
            y_waarden.append(float(waarde))

        fig.add_trace(
            go.Scatter(
                x=x_waarden,
                y=y_waarden,
                mode="lines+markers",
                name="CMP-10",
                line=dict(
                    color="#D69E2E",
                    width=1.8
                ),
                marker=dict(size=3),
                opacity=0.9,
                connectgaps=False,
                hovertemplate=(
                    "<b>CMP-10</b>"
                    "<br>Tijd: %{x|%d-%m-%Y %H:%M}"
                    "<br>Waarde: %{y:.2f}"
                    "<extra></extra>"
                )
            )
        )

    # Overige variabelen per zonnepaneel weergeven
    else:

        for paneel in panelen:

            data = (
                df_tijd[df_tijd["label"] == paneel]
                .sort_values("_time")
                .copy()
            )

            # Alleen beschikbare waarden van deze variabele
            data = data.dropna(subset=[variabele])

            # Meetgaten groter dan 20 minuten bepalen
            tijdverschil = (
                data["_time"]
                .diff()
                .dt.total_seconds()
                .div(60)
            )

            x_waarden = []
            y_waarden = []

            for tijd, waarde, verschil in zip(
                data["_time"],
                data[variabele],
                tijdverschil
            ):

                if pd.notna(verschil) and verschil > 20:
                    x_waarden.append(tijd)
                    y_waarden.append(None)

                x_waarden.append(tijd)
                y_waarden.append(float(waarde))

            paneelnaam = paneel.replace("PvDevice-", "PV")

            # Meetreeks toevoegen
            fig.add_trace(
                go.Scatter(
                    x=x_waarden,
                    y=y_waarden,
                    mode="lines+markers",
                    name=paneelnaam,
                    line=dict(
                        color=paneelkleuren[paneel],
                        width=1.5
                    ),
                    marker=dict(size=4),
                    opacity=0.85,
                    connectgaps=False,
                    hovertemplate=(
                        f"<b>{paneelnaam}</b>"
                        "<br>Tijd: %{x|%d-%m-%Y %H:%M}"
                        "<br>Waarde: %{y:.2f}"
                        "<extra></extra>"
                    )
                )
            )

    # Opmaak van iedere individuele figuur
    fig.update_layout(
        title=dict(
            text=titel,
            x=0.5,
            xanchor="center",
            font=dict(size=20)
        ),
        xaxis_title="Datum (UTC)",
        yaxis_title=eenheid,
        height=450,
        template="plotly_white",
        hovermode="closest",
        showlegend=(variabele != "CMP-10"),
        legend=dict(
            orientation="h",
            x=0.5,
            xanchor="center",
            y=1.02,
            yanchor="bottom"
        ),
        margin=dict(
            l=75,
            r=35,
            t=110,
            b=65
        )
    )

    fig.update_xaxes(
        tickformat="%d-%m\n%H:%M",
        showgrid=True,
        gridcolor="rgba(150,150,150,0.15)"
    )

    fig.update_yaxes(
        showgrid=True,
        gridcolor="rgba(150,150,150,0.2)",
        zeroline=False
    )

    # Elke figuur afzonderlijk weergeven
    fig.show()


# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:

import pandas as pd
from IPython.display import display

weer_variabelen = {
    "Temperature-PVMET500": "Omgevingstemperatuur",
    "Humidity-PVMET500": "Relatieve luchtvochtigheid",
    "Pressure-PVMET500": "Luchtdruk"
}

df_weer = df_weather.copy()

df_weer["_time"] = pd.to_datetime(
    df_weer["_time"], utc=True, errors="coerce"
)

df_weer = df_weer.sort_values("_time")

for kolom in weer_variabelen:
    df_weer[kolom] = pd.to_numeric(
        df_weer[kolom], errors="coerce"
    )

# Beschikbaarheid per variabele
overzicht_weer = []

for kolom, naam in weer_variabelen.items():

    geldig = df_weer.dropna(subset=["_time", kolom])
    interval = geldig["_time"].diff().median()

    overzicht_weer.append({
        "Variabele": naam,
        "Aantal metingen": len(geldig),
        "Eerste meting": geldig["_time"].min(),
        "Laatste meting": geldig["_time"].max(),
        "Mediaan interval (min)": (
            round(interval.total_seconds() / 60, 1)
            if pd.notna(interval) else None
        )
    })

tabel_weer = pd.DataFrame(overzicht_weer)

for kolom in ["Eerste meting", "Laatste meting"]:
    tabel_weer[kolom] = (
        tabel_weer[kolom].dt.strftime("%d-%m-%Y %H:%M")
    )

display(tabel_weer)

# Beschrijvende statistiek
statistiek_weer = (
    df_weer[list(weer_variabelen)]
    .describe()
    .T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
    .round(2)
)

statistiek_weer.index = [
    weer_variabelen[k] for k in statistiek_weer.index
]

display(statistiek_weer)


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(
    1, 3,
    figsize=(13, 4),
    constrained_layout=True
)

for ax, (kolom, naam) in zip(
    axes, weer_variabelen.items()
):
    sns.boxplot(
        y=df_weer[kolom].dropna(),
        ax=ax,
        color="#4682B4",
        width=0.35,
        fliersize=3
    )

    ax.set_title(naam, fontsize=11, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(naam)
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    ax.set_axisbelow(True)
    sns.despine(ax=ax)

fig.suptitle(
    "Verdeling van de weergegevens",
    fontsize=15,
    fontweight="bold"
)

plt.show()


In [ ]:

import plotly.graph_objects as go

weer_grafieken = {
    "Temperature-PVMET500": (
        "Omgevingstemperatuur", "Temperatuur (°C)"
    ),
    "Humidity-PVMET500": (
        "Relatieve luchtvochtigheid", "Luchtvochtigheid (%)"
    ),
    "Pressure-PVMET500": (
        "Luchtdruk", "Luchtdruk (eenheid verifiëren)"
    )
}

kleuren = ["#E65100", "#1565C0", "#2E7D32"]

for (kolom, (titel, eenheid)), kleur in zip(
    weer_grafieken.items(), kleuren
):

    data = (
        df_weer[["_time", kolom]]
        .dropna()
        .sort_values("_time")
        .copy()
    )

    # Meetgaten groter dan 20 minuten onderbreken
    verschil = data["_time"].diff().dt.total_seconds() / 60

    x_waarden = []
    y_waarden = []

    for tijd, waarde, minuten in zip(
        data["_time"], data[kolom], verschil
    ):
        if pd.notna(minuten) and minuten > 20:
            x_waarden.append(tijd)
            y_waarden.append(None)

        x_waarden.append(tijd)
        y_waarden.append(float(waarde))

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=x_waarden,
            y=y_waarden,
            mode="lines+markers",
            name=titel,
            line=dict(color=kleur, width=1.8),
            marker=dict(size=3),
            connectgaps=False,
            hovertemplate=(
                "Tijd: %{x|%d-%m-%Y %H:%M}"
                "<br>Waarde: %{y:.2f}"
                "<extra></extra>"
            )
        )
    )

    fig.update_layout(
        title=dict(text=titel, x=0.5),
        xaxis_title="Datum (UTC)",
        yaxis_title=eenheid,
        height=370,
        template="plotly_white",
        hovermode="closest",
        showlegend=False,
        margin=dict(l=75, r=35, t=65, b=65)
    )

    fig.update_xaxes(tickformat="%d-%m\n%H:%M")

    fig.show()


# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.